# Title: Animal Adoption Prediction

## Summary

Animal adoption is a critical issue for shelters, as many animals experience prolonged stays or never find a permanent home. 
This study aims to build a classification model that predicts whether an animal will be adopted based on its attributes. 
Using data from the [Long Beach Animal Shelter](https://github.com/rfordatascience/tidytuesday/blob/main/data/2025/2025-03-04/readme.md), we will analyze key factors such as species, breed, age, health status, and intake type. 
Through exploratory data analysis and predictive modeling, we hope to identify patterns that influence adoption rates.
Our goal is to provide insights that can help shelters improve adoption strategies and increase successful placements.


## Introduction

### Background Information


Animal shelters play a crucial role in rehoming stray and surrendered animals. However, many animals face extended shelter stays, 
leading to overcrowding and resource constraints ([ASPCA, 2025](https://www.aspca.org/helping-people-pets/shelter-intake-and-surrender/pet-statistics?utm_source=chatgpt.com)). 

Recent studies indicate that **adoption rates vary significantly by species, breed, and medical condition** ([The Zebra, 2025](https://www.thezebra.com/resources/research/pet-adoption-statistics/?utm_source=chatgpt.com)). 
Understanding what factors influence adoption can help shelters make data-driven decisions to improve adoption rates. 

In recent years, data-driven approaches have been applied to predict adoption outcomes, allowing shelters to prioritize resources 
and create targeted adoption campaigns ([BMC Vet Research, 2020](https://bmcvetres.biomedcentral.com/articles/10.1186/s12917-020-02728-2?utm_source=chatgpt.com)). 
Previous studies have shown that characteristics such as species, age, breed, and health status 
can significantly impact adoption likelihood ([DrPress, 2021](https://drpress.org/ojs/index.php/HSET/article/view/28962?utm_source=chatgpt.com)). 
By leveraging machine learning techniques, we aim to develop a model that predicts 
whether an animal will be adopted based on its attributes.

### Problem Statement

Can we predict whether an animal will be adopted based on its attributes? 
This study aims to address this question using a classification model trained on animal shelter intake and outcome data. 
Specifically, we will classify each animal as either **adopted (Yes) or not adopted (No)** using key features such as species, 
age, breed, and medical conditions. 

By understanding which features are most important in predicting adoption, we can provide actionable insights to shelters 
to increase adoption rates and improve animal welfare.


### Dataset Description

The dataset used in this study comes from the **TidyTuesday project** and contains **animal intake and outcome records** from the **Long Beach Animal Shelter**. The dataset provides details on each animal’s characteristics, the reason for intake, and the outcome of their stay at the shelter.

This dataset is publicly available and can be accessed here:  
[Long Beach Animal Shelter Data](https://github.com/rfordatascience/tidytuesday/blob/main/data/2025/2025-03-04/readme.md).

#### **Key Variables**
Below are some of the key variables relevant to predicting adoption outcomes:

- **`animal_id`** → Unique identification number for each animal.  
- **`animal_name`** → Given name of the animal (if available).  
- **`animal_type`** → Type of animal (e.g., Dog, Cat).  
- **`primary_color`**, **`secondary_color`** → Animal’s primary and secondary colors.  
- **`sex`** → Altered sex status of the animal (e.g., Male/Neutered, Female/Spayed).  
- **`dob`** → Date of birth (if available).  
- **`intake_date`** → Date the animal was brought to the shelter.  
- **`intake_condition`** → Condition at intake (e.g., Healthy, Sick, Injured).  
- **`intake_type`** → Reason for intake (e.g., Stray, Owner Surrender, Adoption Return).  
- **`outcome_date`** → Date of the recorded outcome (e.g., adoption, euthanasia).  
- **`outcome_type`** → The final result for the animal (e.g., Adopted, Died, Euthanized, Transferred).  
- **`outcome_is_dead`** → Whether the animal was deceased at the time of outcome (`True/False`).  
- **`was_outcome_alive`** → Whether the animal was alive at outcome (`True/False`).  


#### **Identifying the Target Variable (`adopted`)**
Our project aims to **predict whether an animal is adopted**. However, the dataset does **not have a direct column named `adopted`**. Instead, we need to **derive** this outcome based on the `outcome_type` variable.

**Defining `adopted` as the target variable:**
- If **`outcome_type`** = `"Adopted"`, we define `adopted = Yes`.  
- If **`outcome_type`** is any other category (e.g., `"transferred"`, `"euthanized"`, `"return to owner"`), we define `adopted = No`.  

By structuring the problem in this way, we aim to develop a predictive model that can help shelters identify adoption trends and improve adoption strategies.


---

## Methods & Results:

In [3]:
# Load necessary libraries
library(readr)
library(dplyr)

# Read the dataset
longbeach <- read_csv("data/raw/longbeach.csv")
# Print the first few rows to confirm successful loading
head(longbeach)


Rows: 29787 Columns: 22
-- Column specification --------------------------------------------------------
Delimiter: ","
chr  (15): animal_id, animal_name, animal_type, primary_color, secondary_col...
dbl   (2): latitude, longitude
lgl   (2): outcome_is_dead, was_outcome_alive
date  (3): dob, intake_date, outcome_date

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


animal_id,animal_name,animal_type,primary_color,secondary_color,sex,dob,intake_date,intake_condition,intake_type,...,outcome_date,crossing,jurisdiction,outcome_type,outcome_subtype,latitude,longitude,outcome_is_dead,was_outcome_alive,geopoint
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<date>,<chr>,<chr>,...,<date>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<lgl>,<lgl>,<chr>
A693708,*charlien,dog,white,NA,Female,2013-02-21,2023-02-20,ill mild,stray,...,2023-02-26,"[2600 BLK LONG BEACH BLVD, LONG BEACH CA, 90806",Long Beach,euthanasia,ill severe,33.80479,-118.1889,TRUE,FALSE,"33.8047935, -118.1889261"
A708149,NA,reptile,brown,green,Unknown,NA,2023-10-03,normal,stray,...,2023-10-03,"`600 BLK E HARCOURT, LB 90805",Long Beach,rescue,other resc,33.86800,-118.2009,FALSE,TRUE,"33.8679994, -118.2009307"
A638068,NA,bird,green,red,Unknown,NA,2020-01-01,injured severe,wildlife,...,2020-01-01,"0 BLK GRAND AVE, LONG BEACH, CA 90803",Long Beach,euthanasia,inj severe,33.76048,-118.1481,TRUE,FALSE,"33.7604783, -118.1480912"
A639310,NA,bird,white,gray,Unknown,NA,2020-02-02,ill severe,wildlife,...,2020-02-02,"0 BLK TEMPLE AVE, LONG BEACH, CA 90803",Long Beach,transfer,lbah,33.76246,-118.1597,FALSE,TRUE,"33.7624598, -118.1596777"
A618968,*morgan,cat,black,white,Female,2014-12-18,2018-12-18,injured severe,stray,...,2019-01-13,"0 BLK W ZANE ST, LONG BEACH, CA 90805",Long Beach,rescue,littlelion,33.84950,-118.1949,FALSE,TRUE,"33.8495009, -118.1949053"
A730385,*brandon,rabbit,black,white,Neutered,2023-04-19,2024-10-18,normal,stray,...,2024-11-15,00 AQUARIUM WAY LONG BEACH CA 90802,Long Beach,adoption,web,33.76399,-118.1944,FALSE,TRUE,"33.7639859, -118.1944096"


In [4]:
# View unique values and their counts in outcome_type
outcome_summary <- longbeach %>%
  count(outcome_type, sort = TRUE)

# Print the summary
print(outcome_summary)


# A tibble: 19 x 2
   outcome_type                n
   <chr>                   <int>
 1 rescue                   6680
 2 adoption                 6290
 3 euthanasia               5451
 4 transfer                 4869
 5 return to owner          3214
 6 shelter, neuter, return   919
 7 died                      763
 8 community cat             386
 9 return to wild habitat    294
10 transport                 206
11 NA                        187
12 foster to adopt           166
13 homefirst                  88
14 disposal                   71
15 missing                    59
16 trap, neuter, release      57
17 return to rescue           46
18 duplicate                  31
19 foster                     10


# Discussion:

## References:

1. ASPCA. (2025). Shelter Intake and Adoption Statistics. Retrieved from  
   [https://www.aspca.org/helping-people-pets/shelter-intake-and-surrender/pet-statistics](https://www.aspca.org/helping-people-pets/shelter-intake-and-surrender/pet-statistics)  
2. The Zebra. (2025). Pet Adoption Statistics. Retrieved from  
   [https://www.thezebra.com/resources/research/pet-adoption-statistics](https://www.thezebra.com/resources/research/pet-adoption-statistics)  
3. BMC Veterinary Research. (2020). Increasing Adoption Rates at Animal Shelters: Data-Driven Approaches. Retrieved from  
   [https://bmcvetres.biomedcentral.com/articles/10.1186/s12917-020-02728-2](https://bmcvetres.biomedcentral.com/articles/10.1186/s12917-020-02728-2)  
4. DrPress. (2021). Predicting Pet Adoption Outcomes Using Machine Learning. Retrieved from  
   [https://drpress.org/ojs/index.php/HSET/article/view/28962](https://drpress.org/ojs/index.php/HSET/article/view/28962)  
